# 📄 RFP 문서 검색 시스템 - Retrieval 고도화 V9

## V4 → V9 변경사항

| # | 항목 | V4 | V9 |
|---|---|---|---|
| 1 | 데이터 | advanced 청크 (5,667개) | 원본 파싱 청크 (51,465개) |
| 2 | 메타데이터 | 파일명/파일형식 | + 페이지(PDF) / 섹션명(HWP) |
| 3 | 출처 표시 | [문서명.hwp] | [문서명.pdf, p.3] / [문서명.hwp, §섹션명] |
| 4 | 검색 방식 | Dense only | Dense + BM25 Hybrid (RRF) |
| 5 | Re-ranking | 없음 | BGE-Reranker 적용 |
| 6 | 평가 지표 | Hit@3/5/10 / MRR | + nDCG / Latency |
| 7 | PDF 청크 | 없음 (doc_id 충돌로 삭제) | PDF doc_id 10000번대로 분리 복구 |


---
## 1. 환경 설정

In [1]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from rapidfuzz import process, fuzz

load_dotenv(dotenv_path="/home/chanyoung/.env", override=True)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

BASE_DIR = Path.home() / "AI-based-RFP-RAG-System"
DATA_DIR = BASE_DIR / "data"

print("✅ 환경 설정 완료")
print(f"   BASE_DIR: {BASE_DIR}")

✅ 환경 설정 완료
   BASE_DIR: /home/chanyoung/AI-based-RFP-RAG-System


---
## 2. BM25 라이브러리 설치 및 테스트

In [2]:
import subprocess
import sys

# pcyenv pip으로 설치 (권한 오류 방지)
pcyenv_pip = str(__import__('pathlib').Path.home() / "AI-based-RFP-RAG-System" / "pcyenv" / "bin" / "pip")
subprocess.run([pcyenv_pip, "install", "rank_bm25", "--quiet"])

from rank_bm25 import BM25Okapi
print("✅ BM25 설치 완료")

✅ BM25 설치 완료


---
## 3. BGE-Reranker 설치 및 로드 테스트

In [3]:
import subprocess
from pathlib import Path

pcyenv_pip = str(Path.home() / "AI-based-RFP-RAG-System" / "pcyenv" / "bin" / "pip")
subprocess.run([pcyenv_pip, "install", "sentence-transformers", "--quiet"])

from sentence_transformers import CrossEncoder
import torch

print("⏳ BGE-Reranker 모델 로드 중... (최초 실행 시 다운로드 약 1~2분 소요)")

# 캐시 경로 지정 (재실행 시 다운로드 스킵)
RERANKER_CACHE = str(Path.home() / "AI-based-RFP-RAG-System" / "models" / "bge-reranker")
reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=512)
print("✅ BGE-Reranker 로드 완료")

/home/chanyoung/AI-based-RFP-RAG-System/pcyenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ BGE-Reranker 모델 로드 중... (최초 실행 시 다운로드 약 1~2분 소요)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4069.66it/s]


✅ BGE-Reranker 로드 완료


---
## 4. 원본 파싱 청크 로드

| 파일 | 내용 | 건수 |
|---|---|---|
| `chunks_data.json` | HWP + PDF 통합 청크 | **51,465개** (HWP 44,598 + PDF 6,867) |

### 메타데이터 구조
- HWP: `doc_id`, `사업명`, `발주기관`, `파일형식`, `section`, `chunk_id`
- PDF: `doc_id`, `사업명`, `발주기관`, `파일형식`, `파일명`, `page`, `chunk_id`

### V8 → V9 변경사항
- PDF doc_id 충돌 수정: HWP(0~664) / PDF(10000~10024) 범위 분리
- 삭제됐던 PDF 청크 6,867개 복구
- 전체 청크 49,395개 → **51,465개**


In [4]:
with open(DATA_DIR / "chunks_data.json", "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print(f"✅ 청크 로드 완료: {len(all_chunks)}개")

# 파일형식별 통계
hwp_count = sum(1 for c in all_chunks if c["metadata"].get("파일형식") == "hwp")
pdf_count = sum(1 for c in all_chunks if c["metadata"].get("파일형식") == "pdf")
print(f"   HWP: {hwp_count}개 / PDF: {pdf_count}개")

# 메타데이터 샘플 확인
print(f"\nHWP 샘플 메타데이터: {all_chunks[0]['metadata']}")
pdf_sample = next(c for c in all_chunks if c["metadata"].get("파일형식") == "pdf")
print(f"PDF 샘플 메타데이터: {pdf_sample['metadata']}")

# 발주기관 목록
agency_list = list({c["metadata"].get("발주기관", "") for c in all_chunks if c["metadata"].get("발주기관")})
print(f"\n발주기관 수: {len(agency_list)}개")

✅ 청크 로드 완료: 51465개
   HWP: 44598개 / PDF: 6867개

HWP 샘플 메타데이터: {'doc_id': '0', '사업명': '한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화', '발주기관': '한영대학', '사업금액': '130000000.0', '공고번호': '20241001798', '파일형식': 'hwp', 'has_full_text': True, 'has_table': True, 'chunk_id': '0_0', 'chunk_index': '0', 'section': ''}
PDF 샘플 메타데이터: {'doc_id': '10000', '사업명': '차세대 포털·학사 정보시스템 구축사업', '발주기관': '고려대학교', '사업금액': '11270000000.0', '공고번호': 'nan', '파일형식': 'pdf', '파일명': '고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', 'page': '5', 'has_full_text': True, 'has_table': True, 'chunk_id': 'pdf_10000_0', 'chunk_index': '0'}

발주기관 수: 406개


---
## 5. ChromaDB 재구축 (V8 전용)

> ⚠️ **처음 실행 시에만** 임베딩 생성 및 저장이 수행됩니다. (약 30~60분 소요)
> 이미 저장된 경우 `get_or_create_collection`으로 바로 연결됩니다.

In [5]:
CHROMA_PATH = str(Path.home() / "chroma_data_v9")
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    name="rfp_documents_v9",
    metadata={"hnsw:space": "cosine"}
)
print(f"✅ ChromaDB 연결: {collection.count()}개 저장됨")


✅ ChromaDB 연결: 0개 저장됨


In [6]:
def get_embeddings(texts: list[str]) -> list[list[float]]:
    """OpenAI text-embedding-3-small 임베딩 생성 (배치 처리)"""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [r.embedding for r in response.data]


def build_chroma_from_chunks(chunks, batch_size=100):
    existing = collection.count()
    if existing >= len(chunks):
        print(f"✅ 이미 구축됨: {existing}개 — 스킵")
        return

    print(f"⏳ ChromaDB 구축 시작: {len(chunks)}개 청크")
    start = time.time()

    for i in range(existing, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        texts = [c["text"] for c in batch]
        ids = [c["chunk_id"] for c in batch]
        metas = []
        for c in batch:
            m = {k: str(v) for k, v in c["metadata"].items()}
            metas.append(m)

        embeddings = get_embeddings(texts)
        collection.add(documents=texts, embeddings=embeddings, ids=ids, metadatas=metas)

        if (i // batch_size) % 10 == 0:
            elapsed = time.time() - start
            print(f"  {i+len(batch)}/{len(chunks)} ({elapsed:.0f}s)")

    print(f"✅ ChromaDB 구축 완료: {collection.count()}개")


build_chroma_from_chunks(all_chunks)

⏳ ChromaDB 구축 시작: 51465개 청크
  100/51465 (2s)
  1100/51465 (37s)
  2100/51465 (50s)
  3100/51465 (63s)
  4100/51465 (75s)
  5100/51465 (89s)
  6100/51465 (101s)
  7100/51465 (130s)
  8100/51465 (143s)
  9100/51465 (158s)
  10100/51465 (171s)
  11100/51465 (183s)
  12100/51465 (195s)
  13100/51465 (209s)
  14100/51465 (222s)
  15100/51465 (234s)
  16100/51465 (248s)
  17100/51465 (260s)
  18100/51465 (273s)
  19100/51465 (285s)
  20100/51465 (298s)
  21100/51465 (313s)
  22100/51465 (325s)
  23100/51465 (344s)
  24100/51465 (356s)
  25100/51465 (368s)
  26100/51465 (381s)
  27100/51465 (395s)
  28100/51465 (408s)
  29100/51465 (420s)
  30100/51465 (432s)
  31100/51465 (444s)
  32100/51465 (457s)
  33100/51465 (471s)
  34100/51465 (483s)
  35100/51465 (501s)
  36100/51465 (514s)
  37100/51465 (527s)
  38100/51465 (540s)
  39100/51465 (552s)
  40100/51465 (565s)
  41100/51465 (579s)
  42100/51465 (592s)
  43100/51465 (605s)
  44100/51465 (635s)
  45100/51465 (646s)
  46100/51465 (657s)
  4

---
## 6. BM25 인덱스 구축

In [7]:
import pickle
from pathlib import Path

BM25_CACHE_PATH = Path.home() / "AI-based-RFP-RAG-System" / "data" / "bm25_index.pkl"

if BM25_CACHE_PATH.exists():
    print("⏳ BM25 인덱스 캐시 로드 중...")
    with open(BM25_CACHE_PATH, "rb") as f:
        bm25 = pickle.load(f)
    print(f"✅ BM25 캐시 로드 완료: {bm25.corpus_size}개")
else:
    print("⏳ BM25 인덱스 구축 중...")
    tokenized_corpus = [chunk["text"].split() for chunk in all_chunks]
    bm25 = BM25Okapi(tokenized_corpus)
    with open(BM25_CACHE_PATH, "wb") as f:
        pickle.dump(bm25, f)
    print(f"✅ BM25 인덱스 구축 및 저장 완료: {len(tokenized_corpus)}개 문서")
    print(f"   저장 경로: {BM25_CACHE_PATH}")

⏳ BM25 인덱스 캐시 로드 중...
✅ BM25 캐시 로드 완료: 48262개


---
## 7. 유틸 함수 (출처 반환 - BR-07)

- PDF: `[문서명.pdf, p.3]`
- HWP: `[문서명.hwp, §섹션명]` (섹션 있을 때)

In [8]:
def get_agency_filter(query: str, threshold: int = None):
    """발주기관 fuzzy matching → where 필터 반환
    threshold: None이면 쿼리 길이에 따라 자동 조절
      - 짧은 쿼리(5자 이하): 80 (엄격)
      - 보통 쿼리(6~10자): 70 (기본)
      - 긴 쿼리(11자 이상): 60 (완화)
    """
    if threshold is None:
        if len(query) <= 5:
            threshold = 80
        elif len(query) <= 10:
            threshold = 70
        else:
            threshold = 60
    match = process.extractOne(query, agency_list, scorer=fuzz.partial_ratio)
    if match and match[1] >= threshold:
        return {"발주기관": match[0]}, match[0]
    return None, None


def format_source(metadata: dict) -> str:
    """BR-07: 출처 표시 (파일형식별 다르게)"""
    파일형식 = metadata.get("파일형식", "")
    파일명 = metadata.get("파일명", metadata.get("사업명", "알 수 없음"))
    if 파일형식 == "pdf":
        page = metadata.get("page", "")
        if page:
            return f"[{파일명}, p.{page}]"
        return f"[{파일명}]"
    else:
        section = metadata.get("section", "")
        if section:
            return f"[{파일명}.hwp, §{section}]"
        return f"[{파일명}.hwp]"


def retrieve_topk(query: str, top_k: int = 5, verbose: bool = False):
    where_filter, agency = get_agency_filter(query)
    query_embedding = get_embeddings([query])[0]

    kwargs = {"query_embeddings": [query_embedding], "n_results": top_k, "include": ["documents", "metadatas", "distances"]}
    if where_filter:
        kwargs["where"] = where_filter

    results = collection.query(**kwargs)
    output = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        score = 1 - dist
        output.append({"text": doc, "metadata": meta, "score": score, "source": format_source(meta)})
        if verbose:
            print(f"  score={score:.4f} | {format_source(meta)}")
    return output


print("✅ 유틸 함수 정의 완료")

✅ 유틸 함수 정의 완료


---
## 8. Hybrid Search (Dense + BM25 + RRF)

**RRF (Reciprocal Rank Fusion)**: 두 검색 결과의 순위를 융합하는 알고리즘
- 공식: `RRF(d) = Σ 1 / (k + rank(d))`  (k=60 권장)

In [9]:
def rrf_score(rank: int, k: int = 60) -> float:
    return 1.0 / (k + rank)


def hybrid_search(query: str, top_k: int = 5, alpha: float = 0.5, verbose: bool = False):
    """Hybrid Search: Dense + BM25 + RRF 융합"""
    # 1. Dense 검색 (k=20 후보)
    where_filter, agency = get_agency_filter(query)
    query_embedding = get_embeddings([query])[0]
    kwargs = {"query_embeddings": [query_embedding], "n_results": min(20, collection.count()), "include": ["documents", "metadatas", "distances"]}
    if where_filter:
        kwargs["where"] = where_filter

    dense_results = collection.query(**kwargs)
    dense_docs = []
    for doc, meta, dist in zip(dense_results["documents"][0], dense_results["metadatas"][0], dense_results["distances"][0]):
        dense_docs.append({"text": doc, "metadata": meta, "score": 1 - dist, "source": format_source(meta)})

    # 2. BM25 검색 (k=20 후보)
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)
    top_bm25_idx = np.argsort(bm25_scores)[::-1][:20]
    bm25_docs = []
    for idx in top_bm25_idx:
        chunk = all_chunks[idx]
        bm25_docs.append({"text": chunk["text"], "metadata": {k: str(v) for k, v in chunk["metadata"].items()}, "score": float(bm25_scores[idx]), "source": format_source(chunk["metadata"])})

    # 3. RRF 융합
    rrf_map = {}
    for rank, doc in enumerate(dense_docs):
        key = doc["metadata"].get("chunk_id", doc["text"][:50])
        rrf_map[key] = rrf_map.get(key, {"doc": doc, "rrf": 0})
        rrf_map[key]["rrf"] += rrf_score(rank)

    for rank, doc in enumerate(bm25_docs):
        key = doc["metadata"].get("chunk_id", doc["text"][:50])
        rrf_map[key] = rrf_map.get(key, {"doc": doc, "rrf": 0})
        rrf_map[key]["rrf"] += rrf_score(rank)

    sorted_results = sorted(rrf_map.values(), key=lambda x: x["rrf"], reverse=True)
    output = []
    for item in sorted_results[:top_k]:
        item["doc"]["rrf_score"] = item["rrf"]
        output.append(item["doc"])
        if verbose:
            print(f"  rrf={item['rrf']:.4f} | {item['doc']['source']}")

    return output


print("✅ Hybrid Search (Dense + BM25 + RRF) 정의 완료")

✅ Hybrid Search (Dense + BM25 + RRF) 정의 완료


---
## 9. BGE-Reranker 적용

1차 검색 결과 (k=20) → CrossEncoder로 재정렬 → 상위 k개 반환

In [10]:
def rerank_results(query: str, results: list, top_k: int = 5) -> list:
    """BGE-Reranker로 검색 결과 재정렬"""
    if not results:
        return results

    pairs = [[query, r["text"]] for r in results]
    scores = reranker.predict(pairs)

    for i, r in enumerate(results):
        r["rerank_score"] = float(scores[i])

    reranked = sorted(results, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]


def retrieve_with_rerank(query: str, top_k: int = 5, use_hybrid: bool = True, verbose: bool = False):
    """Hybrid Search + BGE-Reranker 통합 검색"""
    start = time.time()

    # 1차 검색 (k=20)
    if use_hybrid:
        candidates = hybrid_search(query, top_k=20, verbose=False)
    else:
        candidates = retrieve_topk(query, top_k=20, verbose=False)

    # Re-ranking
    reranked = rerank_results(query, candidates, top_k=top_k)
    latency = time.time() - start

    if verbose:
        print(f"  Latency: {latency:.3f}s")
        for i, r in enumerate(reranked):
            print(f"  [{i+1}] rerank={r['rerank_score']:.4f} | {r['source']}")

    return reranked, latency


print("✅ BGE-Reranker 통합 검색 정의 완료")

✅ BGE-Reranker 통합 검색 정의 완료


---
## 10. 검색 전략 통합 (MMR + Query Rewriting + HyDE + Hybrid + Reranker)

In [11]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


def retrieve_mmr(query, top_k=5, fetch_k=20, lambda_mult=0.7, verbose=False):
    query_embedding = get_embeddings([query])[0]
    where_filter, _ = get_agency_filter(query)
    kwargs = {"query_embeddings": [query_embedding], "n_results": min(fetch_k, collection.count()), "include": ["documents", "metadatas", "distances", "embeddings"]}
    if where_filter:
        kwargs["where"] = where_filter
    results = collection.query(**kwargs)

    candidates = []
    for doc, meta, dist, emb in zip(results["documents"][0], results["metadatas"][0], results["distances"][0], results["embeddings"][0]):
        candidates.append({"text": doc, "metadata": meta, "score": 1-dist, "embedding": emb, "source": format_source(meta)})

    selected, selected_embs = [], []
    for _ in range(min(top_k, len(candidates))):
        if not selected:
            best = max(candidates, key=lambda x: x["score"])
        else:
            best = max(candidates, key=lambda x: lambda_mult * x["score"] - (1 - lambda_mult) * max(cosine_similarity(x["embedding"], e) for e in selected_embs))
        selected.append(best)
        selected_embs.append(best["embedding"])
        candidates.remove(best)
    return selected


REWRITE_PROMPT = """다음 질문을 RFP 문서 검색에 최적화된 형태로 재작성하세요.
핵심 키워드(발주기관, 사업명, 금액, 기간 등)를 명확히 포함해주세요.
질문: {query}
재작성된 질문:"""

HYDE_PROMPT = """다음 질문에 대해 RFP 조달 문서에서 찾을 수 있는 내용으로 가상 답변을 작성하세요.
구체적인 숫자, 기관명, 사업명을 포함해 2~3문장으로 작성하세요.
질문: {query}
가상 답변:"""

MULTI_QUERY_PROMPT = """다음 질문을 검색 성능 향상을 위해 3가지 다른 방식으로 확장해주세요.
각 질문은 핵심 의도를 유지하되 다른 표현을 사용하세요.
원본 질문: {query}
확장 질문 3개 (줄바꿈으로 구분):"""


def get_llm_response(prompt: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


def retrieve_with_prompt(query: str, strategy: str = "original", top_k: int = 10, lambda_mult: float = 0.7, verbose: bool = False):
    """전략별 검색 (original/rewrite/hyde/mmr_rewrite/hybrid/hybrid_rerank/multi_query)"""
    if strategy == "original":
        return retrieve_topk(query, top_k=top_k, verbose=verbose)

    elif strategy == "rewrite":
        rewritten = get_llm_response(REWRITE_PROMPT.format(query=query))
        if verbose: print(f"  재작성: {rewritten}")
        return retrieve_topk(rewritten, top_k=top_k, verbose=verbose)

    elif strategy == "hyde":
        hypothesis = get_llm_response(HYDE_PROMPT.format(query=query))
        if verbose: print(f"  가상답변: {hypothesis[:80]}...")
        return retrieve_topk(hypothesis, top_k=top_k, verbose=verbose)

    elif strategy == "mmr_rewrite":
        rewritten = get_llm_response(REWRITE_PROMPT.format(query=query))
        return retrieve_mmr(rewritten, top_k=top_k, lambda_mult=lambda_mult, verbose=verbose)

    elif strategy == "hybrid":
        return hybrid_search(query, top_k=top_k, verbose=verbose)

    elif strategy == "hybrid_rerank":
        reranked, _ = retrieve_with_rerank(query, top_k=top_k, use_hybrid=True, verbose=verbose)
        return reranked

    elif strategy == "multi_query":
        multi = get_llm_response(MULTI_QUERY_PROMPT.format(query=query))
        queries = [query] + [q.strip() for q in multi.strip().split("\n") if q.strip()][:3]
        seen, merged = set(), []
        for q in queries:
            for r in retrieve_topk(q, top_k=top_k//2, verbose=False):
                key = r["metadata"].get("chunk_id", r["text"][:50])
                if key not in seen:
                    seen.add(key)
                    merged.append(r)
        return sorted(merged, key=lambda x: x["score"], reverse=True)[:top_k]

    return retrieve_topk(query, top_k=top_k)


print("✅ 전략 통합 함수 정의 완료")
print("   전략 목록: original / rewrite / hyde / mmr_rewrite / hybrid / hybrid_rerank / multi_query")

✅ 전략 통합 함수 정의 완료
   전략 목록: original / rewrite / hyde / mmr_rewrite / hybrid / hybrid_rerank / multi_query


---
## 11. BR-07 출처 반환 확인

PDF는 페이지, HWP는 섹션명으로 출처 표시됩니다.

In [12]:
test_query = "한국가스공사 차세대 ERP 구축 사업 예산"

print(f"📌 질문: {test_query}\n")
results = retrieve_topk(test_query, top_k=5)

for i, r in enumerate(results):
    print(f"[{i+1}] score={r['score']:.4f}")
    print(f"      출처    : {r['source']}")
    print(f"      사업명  : {r['metadata']['사업명']}")
    print(f"      발주기관: {r['metadata']['발주기관']}")
    print(f"      텍스트  : {r['text'][:80]}...")
    print()

📌 질문: 한국가스공사 차세대 ERP 구축 사업 예산

[1] score=0.5379
      출처    : [[재공고]차세대 통합정보시스템(ERP) 구축.hwp, §2. 사업목적]
      사업명  : [재공고]차세대 통합정보시스템(ERP) 구축
      발주기관: 한국가스공사
      텍스트  : | Ⅰ 개요 |
| --- |
2. 사업목적
○ 기술지원 종료( 27년)에 대비한 ERP 업그레이드
- 제조사(SAP )의 기술지원 종료 이후 ...

[2] score=0.4850
      출처    : [[재공고]차세대 통합정보시스템(ERP) 구축.hwp]
      사업명  : [재공고]차세대 통합정보시스템(ERP) 구축
      발주기관: 한국가스공사
      텍스트  : | 요구사항 고유번호 SFR-015 요구사항 명 구분회계 업무 프로세스 개선 및 시스템 개발 요구사항 상세설명 정의 공사의 구분회계 업무 프로세...

[3] score=0.4796
      출처    : [[재공고]차세대 통합정보시스템(ERP) 구축.hwp]
      사업명  : [재공고]차세대 통합정보시스템(ERP) 구축
      발주기관: 한국가스공사
      텍스트  : | 요구사항 고유번호 CSR-003 요구사항 명 전사 PI 결과에 따른 개선 방안 제시 요구사항 상세설명 정의 시스템 기능 고도화를 위한 개선과...

[4] score=0.4502
      출처    : [[재공고]차세대 통합정보시스템(ERP) 구축.hwp]
      사업명  : [재공고]차세대 통합정보시스템(ERP) 구축
      발주기관: 한국가스공사
      텍스트  : | 요구사항 구분 세부 구분 요구사항 ID 요구사항 명 기능 요구사항 컨버전 SFR-001 SAP S/4 HANA System 컨버전 정의 SF...

[5] score=0.4492
      출처    : [[재공고]차세대 통합정보시스템(ERP) 구축.hwp]
      사업명  : [재공고]차세대 통합정보시스템

---
## 12. Dense only vs Hybrid Search 성능 비교

In [13]:
test_queries = [
    "고려대학교 포털 시스템 구축 사업 금액",
    "AI 기반 시스템 사업자 자격 요건",
    "데이터 마이그레이션 일정 및 기간"
]

print("Dense only vs Hybrid Search 비교\n")
print(f"{'질문':<30} {'Dense Top1':^30} {'Hybrid Top1':^30}")
print("-" * 90)

for q in test_queries:
    dense = retrieve_topk(q, top_k=1)
    hybrid = hybrid_search(q, top_k=1)
    d_src = dense[0]["source"] if dense else "-"
    h_src = hybrid[0]["source"] if hybrid else "-"
    print(f"{q[:28]:<30} {d_src[:28]:^30} {h_src[:28]:^30}")

Dense only vs Hybrid Search 비교

질문                                       Dense Top1                    Hybrid Top1          
------------------------------------------------------------------------------------------
고려대학교 포털 시스템 구축 사업 금액           [고려대학교_차세대 포털·학사 정보시스템 구축사업.   [고려대학교 연구실안전관리시스템 고도화 용역.hwp 
AI 기반 시스템 사업자 자격 요건             [「2024년 전문대학 혁신지원사업」차세대 통합정보   [「2024년 전문대학 혁신지원사업」차세대 통합정보 
데이터 마이그레이션 일정 및 기간              [사업지표분석시스템 구축 용역 (재공고).hwp]    [대전관광공사_(긴급) 대전컨벤션센터(DCC) 대관 


---
## 13. Re-ranking 추가 시 Latency 측정

In [14]:
test_query = "차세대 포털 학사 정보시스템 구축 예산"

print(f"📌 질문: {test_query}\n")

# Dense only
t0 = time.time()
_ = retrieve_topk(test_query, top_k=5)
dense_latency = time.time() - t0

# Hybrid
t0 = time.time()
_ = hybrid_search(test_query, top_k=5)
hybrid_latency = time.time() - t0

# Hybrid + Reranker
_, rerank_latency = retrieve_with_rerank(test_query, top_k=5, use_hybrid=True)

print(f"Dense only     : {dense_latency:.3f}s")
print(f"Hybrid (RRF)   : {hybrid_latency:.3f}s")
print(f"Hybrid+Reranker: {rerank_latency:.3f}s")

# P95 / P99 Latency 측정 (10회 반복)
print("\n⏳ Latency 분포 측정 중 (각 10회)...")
for label, fn in [("Dense only", lambda: retrieve_topk(test_query, top_k=5)),
                   ("Hybrid", lambda: hybrid_search(test_query, top_k=5))]:
    times = []
    for _ in range(10):
        t0 = time.time()
        fn()
        times.append(time.time() - t0)
    times.sort()
    print(f"{label:20} avg={np.mean(times):.3f}s | p95={times[int(len(times)*0.95)]:.3f}s | p99={times[int(len(times)*0.99)]:.3f}s")

📌 질문: 차세대 포털 학사 정보시스템 구축 예산

Dense only     : 0.137s
Hybrid (RRF)   : 0.333s
Hybrid+Reranker: 1.293s

⏳ Latency 분포 측정 중 (각 10회)...
Dense only           avg=0.149s | p95=0.173s | p99=0.173s
Hybrid               avg=0.329s | p95=0.395s | p99=0.395s


---
## 14. PM eval셋 로드

| 유형 | 설명 |
|---|---|
| A | 단일 문서 기반 단순 질문 |
| B | 복수 문서 비교/계산 질문 |
| C | 대화 이력 포함 멀티턴 질문 |
| D | 문서에 없는 정보 질문 (없다고 답해야 정답) |
| E | 오타/비문체 질문 |

In [15]:
EVAL_PATH = Path.home() / "AI-based-RFP-RAG-System" / "data" / "pm_data.xlsx"

eval_df = pd.read_excel(EVAL_PATH)
print(f"✅ eval셋 로드: {len(eval_df)}건")
print(f"   컬럼: {list(eval_df.columns)}")
print(f"\n유형별 분포:")
print(eval_df["유형"].value_counts().sort_index())

✅ eval셋 로드: 500건
   컬럼: ['ID', '유형', '난이도', '질문', '정답', '근거 문서', '메타필터', '대화 이력', '소스 페이지', '추론 과정', '검증 포인트']

유형별 분포:
유형
A    150
B    200
C     50
D     50
E     50
Name: count, dtype: int64


In [16]:
import ast

def parse_eval_row(row):
    근거문서 = row.get("근거 문서", "[]")
    if isinstance(근거문서, str):
        try: 근거문서 = ast.literal_eval(근거문서)
        except: 근거문서 = []
    메타필터 = row.get("메타필터", "{}")
    if isinstance(메타필터, str):
        try: 메타필터 = ast.literal_eval(메타필터)
        except: 메타필터 = {}
    expected_agencies = []
    for doc in 근거문서:
        agency = doc.split("_")[0] if "_" in doc else ""
        if agency: expected_agencies.append(agency)
    return {
        "id": row.get("ID", ""),
        "type": row.get("유형", ""),
        "difficulty": row.get("난이도", ""),
        "question": row.get("질문", ""),
        "answer": row.get("정답", ""),
        "source_docs": 근거문서,
        "meta_filter": 메타필터,
        "expected_agencies": expected_agencies
    }

eval_data = [parse_eval_row(row) for _, row in eval_df.iterrows()]
print(f"✅ eval 파싱 완료: {len(eval_data)}건")
sample = eval_data[0]
print(f"\n샘플:")
print(f"  ID     : {sample['id']}")
print(f"  유형   : {sample['type']}")
print(f"  질문   : {sample['question'][:60]}...")
print(f"  근거문서: {sample['source_docs']}")

✅ eval 파싱 완료: 500건

샘플:
  ID     : Q001
  유형   : A
  질문   : 한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?...
  근거문서: ['한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp']


---
## 15. 정량 평가 함수 (Hit@k / MRR / nDCG)

| 지표 | 설명 |
|---|---|
| Hit@k | 상위 k개 결과 중 정답 문서 포함 여부 |
| MRR | 정답 문서가 처음 등장하는 순위의 역수 평균 |
| nDCG | 정답 순위에 가중치를 준 평가 지표 |

In [17]:
def is_hit(results, source_docs, k=5):
    retrieved_sources = [r["metadata"].get("파일명", "") for r in results[:k]]
    for src in source_docs:
        src_base = src.replace(".hwp", "").replace(".pdf", "").strip()
        for ret in retrieved_sources:
            if src_base in ret or ret in src_base:
                return True
    return False


def get_rr(results, source_docs):
    for rank, r in enumerate(results, 1):
        ret = r["metadata"].get("파일명", "")
        for src in source_docs:
            src_base = src.replace(".hwp", "").replace(".pdf", "").strip()
            if src_base in ret or ret in src_base:
                return 1.0 / rank
    return 0.0


def get_ndcg(results, source_docs, k=10):
    """nDCG@k 계산 (0.0 ~ 1.0 범위로 정규화)
    - DCG: 실제 검색 결과의 순위 가중 점수
    - IDCG: 이상적인 순위일 때의 최대 점수
    - nDCG = DCG / IDCG (항상 0~1 사이)
    """
    dcg, idcg = 0.0, 0.0
    # 각 결과에 대해 관련성(0 또는 1) 계산
    for rank, r in enumerate(results[:k], 1):
        ret = r["metadata"].get("파일명", "")
        rel = 0
        for src in source_docs:
            src_base = src.replace(".hwp", "").replace(".pdf", "").strip()
            if src_base in ret or ret in src_base:
                rel = 1
                break
        dcg += rel / np.log2(rank + 1)
    # IDCG: 정답 문서가 1위~n위에 있을 때의 이상적 점수
    n_relevant = min(len(source_docs), k)
    for rank in range(1, n_relevant + 1):
        idcg += 1 / np.log2(rank + 1)
    # nDCG는 반드시 0~1 범위
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return round(min(ndcg, 1.0), 4)


def evaluate_strategy(strategy, eval_subset, top_k=10, lambda_mult=0.7, verbose=False):
    hits3, hits5, hits10, rrs, ndcgs, latencies = [], [], [], [], [], []

    for item in eval_subset:
        try:
            t0 = time.time()
            if strategy == "hybrid_rerank":
                results, _ = retrieve_with_rerank(item["question"], top_k=top_k, use_hybrid=True)
            else:
                results = retrieve_with_prompt(item["question"], strategy=strategy, top_k=top_k, lambda_mult=lambda_mult, verbose=verbose)
            latencies.append(time.time() - t0)

            hits3.append(is_hit(results, item["source_docs"], k=3))
            hits5.append(is_hit(results, item["source_docs"], k=5))
            hits10.append(is_hit(results, item["source_docs"], k=10))
            rrs.append(get_rr(results, item["source_docs"]))
            ndcgs.append(get_ndcg(results, item["source_docs"], k=10))
        except Exception as e:
            print(f"  ⚠️ 오류 [{item['id']}]: {e}")
            hits3.append(0); hits5.append(0); hits10.append(0)
            rrs.append(0.0); ndcgs.append(0.0); latencies.append(0.0)

    return {
        "strategy": strategy,
        "n": len(eval_subset),
        "Hit@3": round(np.mean(hits3), 4),
        "Hit@5": round(np.mean(hits5), 4),
        "Hit@10": round(np.mean(hits10), 4),
        "MRR": round(np.mean(rrs), 4),
        "nDCG@10": round(np.mean(ndcgs), 4),
        "Latency(s)": round(np.mean(latencies), 3)
    }


print("✅ 정량 평가 함수 정의 완료 (Hit@3/5/10 / MRR / nDCG@10 / Latency)")

✅ 정량 평가 함수 정의 완료 (Hit@3/5/10 / MRR / nDCG@10 / Latency)


---
## 16. 전략별 성능 비교

> ⚠️ API 호출이 발생합니다. `n_sample` 조절하세요.
> - 빠른 테스트: `n_sample=20`
> - 전체 평가: `n_sample=None`

In [18]:
# original / hybrid → 토큰 비용 없으므로 500건 전체 평가
# rewrite / hyde / mmr_rewrite / multi_query → gpt-4o-mini 호출로 30건만
eval_ab = [e for e in eval_data if e["type"] in ["A", "B"] and e["source_docs"]]

print(f"전체 A/B 타입: {len(eval_ab)}건")

strategies_full = ["original", "hybrid"]       # 500건
strategies_sample = ["rewrite", "hyde", "mmr_rewrite", "hybrid_rerank", "multi_query"]  # 30건

eval_results = []

# 500건 평가 (토큰 무료)
eval_full = eval_ab
print(f"\n✅ 전체 평가 ({len(eval_full)}건): {strategies_full}")
for strategy in strategies_full:
    print(f"\n⏳ [{strategy}] 전체 {len(eval_full)}건 평가 중...")
    result = evaluate_strategy(strategy, eval_full, top_k=10)
    eval_results.append(result)
    print(f"   Hit@3={result['Hit@3']:.4f} | Hit@5={result['Hit@5']:.4f} | Hit@10={result['Hit@10']:.4f} | MRR={result['MRR']:.4f} | nDCG={result['nDCG@10']:.4f} | Latency={result['Latency(s)']}s")

# 30건 평가 (토큰 유료)
eval_sample = eval_ab[:30]
print(f"\n✅ 샘플 평가 ({len(eval_sample)}건): {strategies_sample}")
for strategy in strategies_sample:
    print(f"\n⏳ [{strategy}] 샘플 {len(eval_sample)}건 평가 중...")
    result = evaluate_strategy(strategy, eval_sample, top_k=10)
    eval_results.append(result)
    print(f"   Hit@3={result['Hit@3']:.4f} | Hit@5={result['Hit@5']:.4f} | Hit@10={result['Hit@10']:.4f} | MRR={result['MRR']:.4f} | nDCG={result['nDCG@10']:.4f} | Latency={result['Latency(s)']}s")

print("\n✅ 전략별 평가 완료")

전체 A/B 타입: 350건

✅ 전체 평가 (350건): ['original', 'hybrid']

⏳ [original] 전체 350건 평가 중...
   Hit@3=0.9971 | Hit@5=1.0000 | Hit@10=1.0000 | MRR=0.9910 | nDCG=0.9981 | Latency=0.394s

⏳ [hybrid] 전체 350건 평가 중...
   Hit@3=1.0000 | Hit@5=1.0000 | Hit@10=1.0000 | MRR=1.0000 | nDCG=1.0000 | Latency=0.914s

✅ 샘플 평가 (30건): ['rewrite', 'hyde', 'mmr_rewrite', 'hybrid_rerank', 'multi_query']

⏳ [rewrite] 샘플 30건 평가 중...
   Hit@3=0.9667 | Hit@5=0.9667 | Hit@10=1.0000 | MRR=0.9704 | nDCG=0.9759 | Latency=2.618s

⏳ [hyde] 샘플 30건 평가 중...
   Hit@3=1.0000 | Hit@5=1.0000 | Hit@10=1.0000 | MRR=1.0000 | nDCG=1.0000 | Latency=2.841s

⏳ [mmr_rewrite] 샘플 30건 평가 중...
   Hit@3=1.0000 | Hit@5=1.0000 | Hit@10=1.0000 | MRR=0.9833 | nDCG=0.9949 | Latency=2.785s

⏳ [hybrid_rerank] 샘플 30건 평가 중...
   Hit@3=1.0000 | Hit@5=1.0000 | Hit@10=1.0000 | MRR=1.0000 | nDCG=1.0000 | Latency=1.001s

⏳ [multi_query] 샘플 30건 평가 중...
   Hit@3=1.0000 | Hit@5=1.0000 | Hit@10=1.0000 | MRR=0.9778 | nDCG=0.9928 | Latency=5.874s

✅ 전략별 평가 완료


In [19]:
result_df = pd.DataFrame(eval_results)
# nDCG 기준으로 정렬 (hybrid가 더 유의미한 검색 품질 반영)
result_df = result_df.sort_values("nDCG@10", ascending=False)
print(result_df.to_string(index=False))

best = result_df.iloc[0]
print(f"\n🏆 최고 전략: [{best['strategy']}]")
print(f"   Hit@3={best['Hit@3']:.4f} | Hit@5={best['Hit@5']:.4f} | Hit@10={best['Hit@10']:.4f}")
print(f"   MRR={best['MRR']:.4f} | nDCG@10={best['nDCG@10']:.4f} | Latency={best['Latency(s)']}s")

print("\n📊 목표치 달성 여부 (팀 기준 문서 기준)")
print(f"  Hit@3  : {best['Hit@3']:.4f} {'✅' if best['Hit@3'] >= 0.70 else '❌'} (목표: 0.70 이상)")
print(f"  Hit@5  : {best['Hit@5']:.4f} {'✅' if best['Hit@5'] >= 0.80 else '❌'} (목표: 0.80 이상)")
print(f"  Hit@10 : {best['Hit@10']:.4f} {'✅' if best['Hit@10'] >= 0.90 else '❌'} (목표: 0.90 이상)")
print(f"  MRR    : {best['MRR']:.4f} {'✅' if best['MRR'] >= 0.60 else '❌'} (목표: 0.60 이상)")

     strategy   n  Hit@3  Hit@5  Hit@10    MRR  nDCG@10  Latency(s)
       hybrid 350 1.0000 1.0000     1.0 1.0000   1.0000       0.914
hybrid_rerank  30 1.0000 1.0000     1.0 1.0000   1.0000       1.001
         hyde  30 1.0000 1.0000     1.0 1.0000   1.0000       2.841
     original 350 0.9971 1.0000     1.0 0.9910   0.9981       0.394
  mmr_rewrite  30 1.0000 1.0000     1.0 0.9833   0.9949       2.785
  multi_query  30 1.0000 1.0000     1.0 0.9778   0.9928       5.874
      rewrite  30 0.9667 0.9667     1.0 0.9704   0.9759       2.618

🏆 최고 전략: [hybrid]
   Hit@3=1.0000 | Hit@5=1.0000 | Hit@10=1.0000
   MRR=1.0000 | nDCG@10=1.0000 | Latency=0.914s

📊 목표치 달성 여부 (팀 기준 문서 기준)
  Hit@3  : 1.0000 ✅ (목표: 0.70 이상)
  Hit@5  : 1.0000 ✅ (목표: 0.80 이상)
  Hit@10 : 1.0000 ✅ (목표: 0.90 이상)
  MRR    : 1.0000 ✅ (목표: 0.60 이상)


---
## 17. 유형별 세부 분석 (A / B / E 타입)

In [20]:
best_strategy = result_df.iloc[0]["strategy"]

print(f"최고 전략 [{best_strategy}] 유형별 성능")
print("=" * 60)

type_results = []

# A/B/E 타입: Hit@k / MRR / nDCG 정상 평가
for qtype in ["A", "B", "E"]:
    subset = [e for e in eval_data if e["type"] == qtype and e["source_docs"]][:20]
    if not subset:
        continue
    res = evaluate_strategy(best_strategy, subset, top_k=10)
    res["type"] = qtype
    type_results.append(res)
    print(f"  {qtype}타입 ({len(subset)}건): Hit@5={res['Hit@5']:.4f} | MRR={res['MRR']:.4f} | nDCG={res['nDCG@10']:.4f} | Latency={res['Latency(s)']}s")

# C 타입: 대화 이력 포함 멀티턴 → 별도 평가
print("\n  C타입 (대화이력 포함): 별도 평가")
eval_c = [e for e in eval_data if e["type"] == "C" and e["source_docs"]][:10]
if eval_c:
    res_c = evaluate_strategy(best_strategy, eval_c, top_k=10)
    print(f"  C타입 ({len(eval_c)}건): Hit@5={res_c['Hit@5']:.4f} | MRR={res_c['MRR']:.4f} | nDCG={res_c['nDCG@10']:.4f}")

# D 타입: Hallucination 테스트 → Hit@k 제외, 검색 자체 되는지 확인
print("\n  D타입 (Hallucination 테스트): 검색 결과 반환 여부만 확인")
eval_d = [e for e in eval_data if e["type"] == "D"][:10]
d_retrieved = 0
for item in eval_d:
    results = retrieve_with_prompt(item["question"], strategy=best_strategy, top_k=5)
    if results:
        d_retrieved += 1
print(f"  D타입 ({len(eval_d)}건): 검색 결과 반환 {d_retrieved}/{len(eval_d)}건 (정답은 '없음'이어야 함)")

최고 전략 [hybrid] 유형별 성능
  A타입 (20건): Hit@5=1.0000 | MRR=1.0000 | nDCG=1.0000 | Latency=0.667s
  B타입 (20건): Hit@5=1.0000 | MRR=1.0000 | nDCG=1.0000 | Latency=0.777s
  E타입 (20건): Hit@5=1.0000 | MRR=1.0000 | nDCG=1.0000 | Latency=0.623s

  C타입 (대화이력 포함): 별도 평가
  C타입 (10건): Hit@5=1.0000 | MRR=1.0000 | nDCG=1.0000

  D타입 (Hallucination 테스트): 검색 결과 반환 여부만 확인
  D타입 (10건): 검색 결과 반환 10/10건 (정답은 '없음'이어야 함)


---
## 18. V4 vs V8 비교 요약

In [21]:
print("=" * 70)
print("📊 V4 vs V8 비교 (HyDE 전략 기준)")
print("=" * 70)
print(f"{'항목':<20} {'V4 (advanced)':^20} {'V8 (원본파싱)':^20}")
print("-" * 70)
print(f"{'청크 수':<20} {'5,667개':^20} {'49,395개':^20}")
print(f"{'데이터 품질':<20} {'낮음 (HWP누락)':^20} {'높음 (원본파싱)':^20}")
print(f"{'출처 표시':<20} {'[문서명.hwp]':^20} {'[문서명.pdf, p.N]':^20}")
print(f"{'검색 방식':<20} {'Dense only':^20} {'Dense+BM25+RRF':^20}")
print(f"{'Re-ranking':<20} {'없음':^20} {'BGE-Reranker':^20}")
print()

v4_best = {"Hit@3": 1.0, "Hit@5": 1.0, "Hit@10": 1.0, "MRR": 0.9778}
v8_best = result_df.iloc[0]
print(f"{'지표':<15} {'V4':^15} {'V8':^15} {'변화':^15}")
print("-" * 60)
for metric in ["Hit@3", "Hit@5", "Hit@10", "MRR"]:
    v4_val = v4_best[metric]
    v8_val = v8_best[metric]
    diff = v8_val - v4_val
    arrow = "▲" if diff > 0 else ("▼" if diff < 0 else "→")
    print(f"  {metric:<13} {v4_val:.4f}{'':^10} {v8_val:.4f}{'':^10} {arrow} {abs(diff):.4f}")

📊 V4 vs V8 비교 (HyDE 전략 기준)
항목                      V4 (advanced)          V8 (원본파싱)      
----------------------------------------------------------------------
청크 수                        5,667개              49,395개       
데이터 품질                    낮음 (HWP누락)           높음 (원본파싱)      
출처 표시                     [문서명.hwp]          [문서명.pdf, p.N]   
검색 방식                     Dense only         Dense+BM25+RRF   
Re-ranking                    없음              BGE-Reranker    

지표                    V4              V8              변화       
------------------------------------------------------------
  Hit@3         1.0000           1.0000           → 0.0000
  Hit@5         1.0000           1.0000           → 0.0000
  Hit@10        1.0000           1.0000           → 0.0000
  MRR           0.9778           1.0000           ▲ 0.0222


---
## 19. 정확도 vs 토큰 비용 트레이드오프 측정

| 전략 | API 호출 | 토큰 비용 |
|---|---|---|
| original | ❌ | 0원 |
| hybrid | ❌ | 0원 |
| hybrid_rerank | ❌ | 0원 (로컬 BGE) |
| rewrite | ✅ | gpt-4o-mini 1회 |
| hyde | ✅ | gpt-4o-mini 1회 |
| mmr_rewrite | ✅ | gpt-4o-mini 1회 |
| multi_query | ✅✅✅ | gpt-4o-mini 3회 |

In [22]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")
test_query = "한국가스공사 차세대 ERP 구축 사업 예산이 얼마인가요?"

rewrite_tokens = len(enc.encode(REWRITE_PROMPT.format(query=test_query)))
hyde_tokens = len(enc.encode(HYDE_PROMPT.format(query=test_query)))
multi_tokens = len(enc.encode(MULTI_QUERY_PROMPT.format(query=test_query))) * 3

input_price = 0.15 / 1_000_000
output_price = 0.60 / 1_000_000
avg_output = 100

print("📊 전략별 토큰 비용 추정 (질문 1건 기준, gpt-4o-mini)")
print("=" * 55)
print(f"{'전략':<15} {'입력토큰':>8} {'비용(원)':>10} {'비고'}")
print("-" * 55)

strategies_cost = [
    ("original",       0,              "API 호출 없음"),
    ("hybrid",         0,              "API 호출 없음"),
    ("hybrid_rerank",  0,              "로컬 BGE 모델"),
    ("rewrite",        rewrite_tokens, "1회 호출"),
    ("hyde",           hyde_tokens,    "1회 호출"),
    ("mmr_rewrite",    rewrite_tokens, "1회 호출"),
    ("multi_query",    multi_tokens,   "3회 호출"),
]

for name, tokens, note in strategies_cost:
    cost_krw = tokens * input_price * 1350 + avg_output * output_price * 1350 if tokens > 0 else 0
    print(f"  {name:<13} {tokens:>8} {cost_krw:>9.4f}원  {note}")

print()
print("💡 500건 전체 평가 시 예상 비용:")
for name, tokens, note in strategies_cost:
    if tokens > 0:
        cost_krw = (tokens * input_price + avg_output * output_price) * 500 * 1350
        print(f"  {name:<13}: 약 {cost_krw:.0f}원")

📊 전략별 토큰 비용 추정 (질문 1건 기준, gpt-4o-mini)
전략                  입력토큰      비용(원) 비고
-------------------------------------------------------
  original             0    0.0000원  API 호출 없음
  hybrid               0    0.0000원  API 호출 없음
  hybrid_rerank        0    0.0000원  로컬 BGE 모델
  rewrite             72    0.0956원  1회 호출
  hyde                75    0.0962원  1회 호출
  mmr_rewrite         72    0.0956원  1회 호출
  multi_query        216    0.1247원  3회 호출

💡 500건 전체 평가 시 예상 비용:
  rewrite      : 약 48원
  hyde         : 약 48원
  mmr_rewrite  : 약 48원
  multi_query  : 약 62원


---
## 20. 최적 k값 결정 (k=3/5/10 비교)

In [23]:
eval_ab_small = [e for e in eval_data if e["type"] in ["A", "B"] and e["source_docs"]][:30]

print("📊 k값별 Hit Rate 비교 (original 전략, 30건)")
print("=" * 50)
print(f"{'k값':>5} {'Hit@k':>8} {'MRR':>8} {'평균토큰수':>12} {'비고'}")
print("-" * 50)

for k in [3, 5, 10]:
    hits, rrs, token_counts = [], [], []
    for item in eval_ab_small:
        results = retrieve_topk(item["question"], top_k=k)
        hits.append(is_hit(results, item["source_docs"], k=k))
        rrs.append(get_rr(results, item["source_docs"]))
        total_tokens = sum(len(r["text"].split()) for r in results)
        token_counts.append(total_tokens)
    note = "⭐ 추천" if k == 5 else ""
    print(f"  k={k:<3} {np.mean(hits):>8.4f} {np.mean(rrs):>8.4f} {np.mean(token_counts):>12.0f}   {note}")

print()
print("💡 k=5 추천: Hit Rate와 토큰 비용의 균형점")

📊 k값별 Hit Rate 비교 (original 전략, 30건)
   k값    Hit@k      MRR        평균토큰수 비고
--------------------------------------------------
  k=3     1.0000   0.9778          688   
  k=5     1.0000   0.9778         1115   ⭐ 추천
  k=10    1.0000   0.9778         2275   

💡 k=5 추천: Hit Rate와 토큰 비용의 균형점


---
## 21. MMR 다양성 확보 여부 확인

In [24]:
test_query = "AI 기반 시스템 구축 사업 요건"
print(f"📌 질문: {test_query}\n")

dense_results = retrieve_topk(test_query, top_k=5)
mmr_results = retrieve_mmr(test_query, top_k=5, lambda_mult=0.7)

print("=== Dense 검색 결과 ===")
dense_docs = []
for i, r in enumerate(dense_results):
    doc_name = r["metadata"].get("사업명", "")[:20]
    dense_docs.append(doc_name)
    print(f"  [{i+1}] {doc_name}")

print("\n=== MMR 검색 결과 (다양성 확보) ===")
mmr_docs = []
for i, r in enumerate(mmr_results):
    doc_name = r["metadata"].get("사업명", "")[:20]
    mmr_docs.append(doc_name)
    print(f"  [{i+1}] {doc_name}")

print(f"\n📊 다양성 비교:")
print(f"  Dense : {len(set(dense_docs))}/5개 고유 문서")
print(f"  MMR   : {len(set(mmr_docs))}/5개 고유 문서")

print("\n📊 lambda 파라미터별 다양성:")
for lmbda in [0.3, 0.5, 0.7, 1.0]:
    results = retrieve_mmr(test_query, top_k=5, lambda_mult=lmbda)
    unique = len(set(r["metadata"].get("사업명", "")[:20] for r in results))
    print(f"  lambda={lmbda}: {unique}/5개 고유 문서")

📌 질문: AI 기반 시스템 구축 사업 요건

=== Dense 검색 결과 ===
  [1] 사명대사공원 미디어아트 콘텐츠 제작 
  [2] 차세대 사업관리시스템 2차년도 인프라
  [3] 곡성군 신청사 정보통신시스템 이전 및
  [4] 2023년 RIPC 통합관리시스템 기
  [5] (협상에 의한 계약)안성시 민원상담콜

=== MMR 검색 결과 (다양성 확보) ===
  [1] 사명대사공원 미디어아트 콘텐츠 제작 
  [2] [입찰공고] 2024~2025학년도 
  [3] 차세대 사업관리시스템 2차년도 인프라
  [4] 엔지니어링공제조합 스마트 금융시스템 
  [5] 곡성군 신청사 정보통신시스템 이전 및

📊 다양성 비교:
  Dense : 5/5개 고유 문서
  MMR   : 5/5개 고유 문서

📊 lambda 파라미터별 다양성:
  lambda=0.3: 5/5개 고유 문서
  lambda=0.5: 5/5개 고유 문서
  lambda=0.7: 5/5개 고유 문서
  lambda=1.0: 5/5개 고유 문서


In [25]:
print("=" * 60)
print("✅ Retrieval 고도화 V9 완료")
print("=" * 60)
print(f"총 청크 수     : {len(all_chunks)}개 (HWP + PDF 통합)")
print(f"Vector DB      : ChromaDB (cosine similarity / HNSW) - V9 전용")
print(f"임베딩 모델    : text-embedding-3-small")
print(f"검색 전략      : Dense / Hybrid(BM25+RRF) / Reranker(BGE)")
print(f"eval셋         : {len(eval_data)}건 (PM 배치 1~25)")
print(f"")
print(f"[V9 변경사항 (V8 대비)]")
print(f"  데이터        : PDF doc_id 충돌 수정 → 51,465개 (PDF 청크 복구)")
print(f"  HWP doc_id    : 0 ~ 664")
print(f"  PDF doc_id    : 10000 ~ 10024")
print(f"  ChromaDB      : chroma_data_v9 / rfp_documents_v9")
print(f"")
print(f"[유지된 기능]")
print(f"  출처 표시     : PDF 페이지 번호 / HWP 섹션명")
print(f"  Hybrid Search : BM25 + Dense + RRF")
print(f"  BGE-Reranker  : BAAI/bge-reranker-base")
print(f"  Multi-Query   : 질문 3~5개 확장 검색")
print(f"  평가 지표     : Hit@3/5/10 / MRR / nDCG@10 / Latency")


✅ Retrieval 고도화 V9 완료
총 청크 수     : 51465개 (HWP + PDF 통합)
Vector DB      : ChromaDB (cosine similarity / HNSW) - V9 전용
임베딩 모델    : text-embedding-3-small
검색 전략      : Dense / Hybrid(BM25+RRF) / Reranker(BGE)
eval셋         : 500건 (PM 배치 1~25)

[V9 변경사항 (V8 대비)]
  데이터        : PDF doc_id 충돌 수정 → 51,465개 (PDF 청크 복구)
  HWP doc_id    : 0 ~ 664
  PDF doc_id    : 10000 ~ 10024
  ChromaDB      : chroma_data_v9 / rfp_documents_v9

[유지된 기능]
  출처 표시     : PDF 페이지 번호 / HWP 섹션명
  Hybrid Search : BM25 + Dense + RRF
  BGE-Reranker  : BAAI/bge-reranker-base
  Multi-Query   : 질문 3~5개 확장 검색
  평가 지표     : Hit@3/5/10 / MRR / nDCG@10 / Latency
